# Machine Learning Tornado prediction model 
This project is a passion project using the open source TorNET dataset developed by MIT in order to create a machine learning based tornado prediction algorithm

<p>Understanding of TorNET file: 
Example file: NUL_210101_195824_KJGX_1085090n_K0.nc<br>
"NUL" = Non-tornadic; "TOR" = tornadic<br>
210101 : date (YY:MM:DD) - 2021, 1, 1 or Jan 1st 2021<br>
195824 : time (UTC) (HR:MIN:SEC) - 19:58:24 UTC<br>
KJGX : NEXRAD radar station (KJGX = Georgia)<br>
1085090n : Event ID<br>
K0 : Random ID<br>
.nc : netCDF filetype<p>

Each TorNet file containes a attribute called "frame_labels"<br>
frame labels is used to identify using 0 and 1 whether or not the storm is tornadic or not<br>
The frame labels looks similar to [0,0,0,0], each number represent a singular frame within the netCDF file<br>
- This understanding of frame_labels will allow us to train the AI model to eventually predict weather the last frame in frame_labels will be a 0 (no tornado) or 1 (tornado) 

In [1]:
# Install necessary Packages
import torch 
import pyart
import numpy as np
import pandas as pd
import xarray as xr
import boto3 
import os


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [2]:
# Opening one of the netcdf files from TorNET
ds = xr.open_dataset('/Users/epalmer/Ethan-s-Meteorology-project-/data/raw/train/2021/NUL_210101_195824_KJGX_1085090n_K0.nc')
print(ds)

print("frame_labels:", ds['frame_labels'].values)

# Find a TOR file 
train_dir = '/Users/epalmer/Ethan-s-Meteorology-project-/data/raw/train/2021'
tor_file = [f for f in os.listdir(train_dir) if f.startswith('TOR')][0]
print("Found:", tor_file)

ds = xr.open_dataset(os.path.join(train_dir, tor_file))
print("frame_labels:", ds['frame_labels'].values)

<xarray.Dataset> Size: 6MB
Dimensions:            (azimuth: 120, range: 240, time: 4, sweep: 2, lims: 2)
Coordinates:
  * azimuth            (azimuth) float32 480B -6.75 -6.25 -5.75 ... 52.25 52.75
  * range              (range) float32 960B 9.05e+04 9.076e+04 ... 1.503e+05
  * time               (time) datetime64[ns] 32B 2021-01-01T19:43:00 ... 2021...
Dimensions without coordinates: sweep, lims
Data variables:
    elevation          (sweep) float64 16B ...
    frame_labels       (time) uint8 4B ...
    nyquist_velocity   (time, sweep) float32 32B ...
    azimuth_limits     (lims) float32 8B ...
    range_limits       (lims) float32 8B ...
    DBZ                (time, azimuth, range, sweep) float32 922kB ...
    VEL                (time, azimuth, range, sweep) float32 922kB ...
    KDP                (time, azimuth, range, sweep) float32 922kB ...
    RHOHV              (time, azimuth, range, sweep) float32 922kB ...
    ZDR                (time, azimuth, range, sweep) float32 922kB 

# Phase one: 
Create a model to detect a tornado (or not) given a current radar snapshot of either a tornado occuring or not occuring

In [3]:

#Importing further packages
import sys
sys.path.append('/Users/epalmer/Ethan-s-Meteorology-project-')

from src.data.dataset import TorNETDatabase #package created 
train_dir = '/Users/epalmer/Ethan-s-Meteorology-project-/data/raw/train/2021'

dataset = TorNETDatabase(train_dir)

print(f"Dataset size: {len(dataset)}")

x, y = dataset[0]
print(f"Input shape: {x.shape}") 
print(f"Label: {y}")     

found19049 samples in /Users/epalmer/Ethan-s-Meteorology-project-/data/raw/train/2021
Dataset size: 19049
Input shape: torch.Size([13, 120, 240])
Label: 0.0


In [4]:
# Finding first Tornadic file in dataset
for i in range(len(dataset)):
    x,y = dataset[i]
    if y == 1.0:
        print (f'First Tornadic Dataset at index: {i}')
        print (f'input shape: {x.shape}')
        print (f'label: {y}')
        break

KeyboardInterrupt: 

In [8]:
# Sanity check for CNN model
from src.models.cnn import TornadoCNN
import inspect

print(inspect.getsource(TornadoCNN))

model = TornadoCNN(in_channels = 13)
dummy = torch.rand(8, 13, 120, 240)
out = model(dummy)
print(f"Output shape: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

class TornadoCNN (nn.Module):
    def __init__(self, in_channels=13):
        super(TornadoCNN, self).__init__()

        self.features = nn.Sequential(
            #Block 1, earlier scan (32, 60, 120)
            nn.Conv2d(in_channels, 32, kernel_size=3, padding = 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace = True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            #Block 2, middle scan (64, 30, 60)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            #Block 3, last scan (128, 15, 30)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
    